# 06 — Feature Assembly & Labelling

This notebook runs the feature assembly pipeline (`app.features`) and
rainfall-threshold labelling (`app.labelling`) to produce a model-ready
training set.

**What it does:**
1. Joins the 1 km grid → weather-point mapping → hourly weather into a
   tidy feature table (monsoon-filtered: May–Oct only).
2. Labels cell-hours as positive when rainfall breaches an
   intensity–duration threshold (Dikshit & Satyam 2019).
3. Samples negatives with ≥ 1 km spatial separation from positives.
4. Plots the spatial distribution of positive cells.

**Terrain note:** `terrain_features.parquet` is produced by a teammate.
When absent, labelling uses rainfall-only (no slope filter) — all cells
show positives because every weather point had ≥ 1 extreme event across
8 monsoons. Re-run after terrain data lands to see proper hill clustering.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from app.features import build_feature_table
from app.labelling import label_by_rainfall_threshold, sample_negatives

DATA_DIR = "../data/processed"

## 1. Build Feature Table (monsoon-only)

In [ ]:
feature_df = build_feature_table(DATA_DIR, monsoon_only=True)

print(f"Shape:           {feature_df.shape}")
print(f"Columns:         {list(feature_df.columns)}")
print(f"Row count:       {len(feature_df):,}")
print(f"Grid cells:      {feature_df['grid_id'].nunique()}")
print(f"Timestamps:      {feature_df['timestamp'].nunique()}")
print(f"Date range:      {feature_df['timestamp'].min()} to {feature_df['timestamp'].max()}")
print(f"Memory:          {feature_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

feature_df.head()

## 2. Label by Rainfall Threshold

**Threshold (Dikshit & Satyam 2019, Darjeeling Himalayas):**
- 1-hour intensity ≥ 20 mm/hr, OR
- 3-hour cumulative ≥ 60 mm

When terrain data is present, an additional slope ≥ 15° filter is applied.
Without terrain, rainfall-only labelling is used (over-represents flat areas).

In [ ]:
labelled_df = label_by_rainfall_threshold(feature_df)

n_pos = int(labelled_df["target_event"].sum())
n_total = len(labelled_df)
pos_rate = n_pos / n_total

print(f"Total rows:      {n_total:,}")
print(f"Positives:       {n_pos:,}")
print(f"Negatives:       {n_total - n_pos:,}")
print(f"Positive rate:   {100*pos_rate:.4f}%")
print(f"Ratio (neg:pos): 1:{(n_total-n_pos)/n_pos:.0f}" if n_pos > 0 else "No positives")
print(f"Terrain filter:  {labelled_df['terrain_filter_applied'].iloc[0]}")

## 3. Spatial Distribution of Positive Cells

In [ ]:
# Count positive hours per grid cell
pos_per_cell = (
    labelled_df[labelled_df["target_event"] == 1]
    .groupby("grid_id")
    .size()
    .reset_index(name="n_positive_hours")
)
print(f"Grid cells with >= 1 positive: {len(pos_per_cell)} / {feature_df['grid_id'].nunique()}")
print(f"\nPositive hours per cell (among affected cells):")
print(pos_per_cell["n_positive_hours"].describe())

In [ ]:
# Load grid geometry and merge positive counts
grid_gdf = gpd.read_parquet(f"{DATA_DIR}/kamrup_metro_grid_1km.parquet")
grid_plot = grid_gdf.merge(pos_per_cell, on="grid_id", how="left")
grid_plot["n_positive_hours"] = grid_plot["n_positive_hours"].fillna(0)
grid_plot["has_positive"] = grid_plot["n_positive_hours"] > 0

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: heatmap of positive count
ax = axes[0]
grid_plot.plot(
    column="n_positive_hours",
    cmap="YlOrRd",
    linewidth=0.3,
    edgecolor="gray",
    legend=True,
    ax=ax,
)
ax.set_title("Positive cell-hours per grid cell\n(rainfall-only, no terrain filter)", fontsize=11)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Right: binary has/doesn't have positives
ax = axes[1]
colors = {True: "#d32f2f", False: "#81c784"}
grid_plot["color"] = grid_plot["has_positive"].map(colors)
grid_plot.plot(color=grid_plot["color"], linewidth=0.3, edgecolor="gray", ax=ax)
legend_handles = [
    mpatches.Patch(color="#d32f2f", label="Has positive events"),
    mpatches.Patch(color="#81c784", label="No positive events"),
]
ax.legend(handles=legend_handles, loc="lower left")
ax.set_title("Grid cells with >= 1 positive event\n(will cluster in hilly areas once terrain filter is applied)", fontsize=11)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig("../docs/img/06_positive_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Sample Negatives (1 km spatial separation)

In [ ]:
training_df = sample_negatives(
    labelled_df, grid_gdf,
    min_separation_m=1000,
    neg_to_pos_ratio=10,
)

n_pos_train = int(training_df["target_event"].sum())
n_neg_train = len(training_df) - n_pos_train

print(f"Training set:    {training_df.shape}")
print(f"Positives:       {n_pos_train:,}")
print(f"Negatives:       {n_neg_train:,}")
print(f"Positive rate:   {100*n_pos_train/len(training_df):.2f}%")
print(f"Ratio (neg:pos): 1:{n_neg_train/n_pos_train:.1f}")
print(f"\nRecommended XGBoost scale_pos_weight: {n_neg_train/n_pos_train:.1f}")

## 5. Summary

| Metric | Value |
|---|---|
| Monsoon-filtered rows | 31,936,512 |
| Grid cells | 904 |
| Monsoon timestamps | 35,328 (May-Oct 2018-2025) |
| Threshold | 20 mm/hr OR 60 mm/3hr (Dikshit & Satyam 2019) |
| Terrain filter | Not applied (awaiting teammate's data) |
| Positives (full table) | 11,785 (0.037%) |
| Full ratio (neg:pos) | 1:2,709 |
| Training set (10:1 sample) | 129,635 (11,785 pos + 117,850 neg) |
| Training positive rate | 9.09% |

**Spatial note:** Without terrain filter, all 904 cells show positives
because each cell inherits its weather point's extreme events over 8
monsoon seasons. The left heatmap shows variation driven by weather-point
zones (19 weather points at ~9 km spacing). Once `terrain_features.parquet`
lands, positives will cluster in hilly areas (slope >= 15 deg).